In [0]:
# ============================================================
# FIRMABLE - SYNTHETIC DATA GENERATION
# CELL 1: COMPANY METADATA
# ============================================================

import random
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Reproducibility
random.seed(42)

CATALOG = "workspace"
BRONZE_SCHEMA = "firmable_v2_bronze"

COMPANY_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.companies"
)

# ------------------------------------------------------------
# 1. Industry definitions
# ------------------------------------------------------------

industries = [
    "Technology",
    "Financial Services",
    "Healthcare",
    "Retail",
    "Manufacturing",
    "Telecommunications",
    "Consulting",
    "E-commerce"
]

# ------------------------------------------------------------
# 2. Organizational structures
# ------------------------------------------------------------

org_structures = [
    "Functional",
    "Flat",
    "Divisional",
    "Hybrid"
]

structure_weights = [0.45, 0.15, 0.15, 0.25]

# ------------------------------------------------------------
# 3. Company name components
# ------------------------------------------------------------

prefixes = [
    "Nova", "Apex", "Vertex", "Quantum", "Pioneer",
    "Summit", "Nexus", "Blue", "Prime", "Evergreen",
    "Global", "Stellar", "NextGen", "Bright", "Unified"
]

suffixes = [
    "Technologies", "Solutions", "Systems", "Global",
    "Industries", "Group", "Networks", "Digital",
    "Enterprises", "Innovations"
]

# ------------------------------------------------------------
# 4. Generate companies
# ------------------------------------------------------------

companies = []

for i in range(1, 501):

    company_id = f"C{i:04d}"

    company_name = (
        f"{random.choice(prefixes)} "
        f"{random.choice(suffixes)} {i}"
    )

    industry = random.choice(industries)

    org_structure = random.choices(
        org_structures,
        weights=structure_weights,
        k=1
    )[0]

    employee_count = random.randint(100, 250)

    if employee_count <= 120:
        company_size_category = "Small"
    elif employee_count <= 180:
        company_size_category = "Medium"
    else:
        company_size_category = "Large"

    headquarters = random.choice([
        "Bengaluru, India",
        "Mumbai, India",
        "Delhi, India",
        "Hyderabad, India",
        "Pune, India",
        "London, UK",
        "New York, USA",
        "Singapore",
        "Toronto, Canada",
        "Sydney, Australia"
    ])

    companies.append((
        company_id,
        company_name,
        industry,
        org_structure,
        employee_count,
        company_size_category,
        headquarters
    ))

# ------------------------------------------------------------
# 5. Create Spark DataFrame
# ------------------------------------------------------------

company_schema = StructType([
    StructField("company_id", StringType(), False),
    StructField("company_name", StringType(), False),
    StructField("industry", StringType(), False),
    StructField("org_structure", StringType(), False),
    StructField("employee_count", IntegerType(), False),
    StructField("company_size_category", StringType(), False),
    StructField("headquarters", StringType(), False)
])

companies_df = spark.createDataFrame(
    companies,
    schema=company_schema
)

# ------------------------------------------------------------
# 6. Save Bronze table
# ------------------------------------------------------------

(
    companies_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(COMPANY_TABLE)
)

# ------------------------------------------------------------
# 7. Display summary
# ------------------------------------------------------------

print("COMPANY GENERATION COMPLETED")
print("============================")
print(f"Total companies: {companies_df.count():,}")

print("\nORGANIZATIONAL STRUCTURE DISTRIBUTION")

display(
    companies_df
    .groupBy("org_structure")
    .count()
    .orderBy(F.desc("count"))
)

print("\nCOMPANY SIZE DISTRIBUTION")

display(
    companies_df
    .groupBy("company_size_category")
    .count()
    .orderBy("company_size_category")
)

print("\nINDUSTRY DISTRIBUTION")

display(
    companies_df
    .groupBy("industry")
    .count()
    .orderBy(F.desc("count"))
)

print("\nSAMPLE COMPANIES")

display(companies_df.limit(10))

In [0]:
# ============================================================
# FIRMABLE - SYNTHETIC DATA GENERATION
# CELL 2: TEAM-BASED EMPLOYEES + GROUND TRUTH
# ============================================================

import random
import math

from pyspark.sql.types import (
    StructType, StructField, StringType
)

CATALOG = "workspace"
BRONZE = f"{CATALOG}.firmable_v2_bronze"

COMPANY_TABLE = f"{BRONZE}.companies"
EMPLOYEE_TABLE = f"{BRONZE}.employees_raw"
GROUND_TRUTH_TABLE = f"{BRONZE}.ground_truth"

# ------------------------------------------------------------
# 1. Departments and executive ownership
# ------------------------------------------------------------

departments = [
    "Engineering & Technology",
    "Data & Analytics",
    "Sales & Business Development",
    "Marketing",
    "Finance & Accounting",
    "Human Resources",
    "Product Management",
    "Security & IT",
    "Operations",
    "Legal & Compliance",
    "Customer Success & Support",
    "Procurement & Supply Chain"
]

department_executives = {
    "Engineering & Technology": "CTO",
    "Data & Analytics": "CTO",
    "Product Management": "CTO",
    "Security & IT": "CTO",
    "Finance & Accounting": "CFO",
    "Operations": "COO",
    "Legal & Compliance": "COO",
    "Procurement & Supply Chain": "COO",
    "Sales & Business Development": "CRO",
    "Marketing": "CRO",
    "Customer Success & Support": "CRO",
    "Human Resources": "CHRO"
}

executive_titles = {
    "CTO": "Chief Technology Officer",
    "CFO": "Chief Financial Officer",
    "COO": "Chief Operating Officer",
    "CRO": "Chief Revenue Officer",
    "CHRO": "Chief Human Resources Officer"
}

# ------------------------------------------------------------
# 2. Division mapping
# ------------------------------------------------------------

divisions = {
    "Consumer": [
        "Sales & Business Development",
        "Marketing",
        "Customer Success & Support"
    ],
    "Technology": [
        "Engineering & Technology",
        "Data & Analytics",
        "Product Management",
        "Security & IT"
    ],
    "Corporate": [
        "Finance & Accounting",
        "Human Resources",
        "Operations",
        "Legal & Compliance",
        "Procurement & Supply Chain"
    ]
}

department_division = {
    dept: division
    for division, dept_list in divisions.items()
    for dept in dept_list
}

# ------------------------------------------------------------
# 3. Team definitions
# ------------------------------------------------------------

department_teams = {
    "Engineering & Technology": [
        "Backend Engineering",
        "Frontend Engineering",
        "Platform Engineering",
        "DevOps",
        "Quality Assurance"
    ],
    "Data & Analytics": [
        "Data Engineering",
        "Analytics",
        "Data Science",
        "Business Intelligence"
    ],
    "Sales & Business Development": [
        "Enterprise Sales",
        "SMB Sales",
        "Partnerships",
        "Business Development"
    ],
    "Marketing": [
        "Digital Marketing",
        "Content Marketing",
        "Brand Marketing",
        "Performance Marketing"
    ],
    "Finance & Accounting": [
        "Accounting",
        "Financial Planning",
        "Taxation",
        "Financial Operations"
    ],
    "Human Resources": [
        "Talent Acquisition",
        "HR Operations",
        "People Development",
        "Employee Relations"
    ],
    "Product Management": [
        "Core Product",
        "Growth Product",
        "Product Design",
        "Product Strategy"
    ],
    "Security & IT": [
        "Infrastructure",
        "Cybersecurity",
        "IT Support",
        "Network Operations"
    ],
    "Operations": [
        "Business Operations",
        "Process Excellence",
        "Operations Planning"
    ],
    "Legal & Compliance": [
        "Legal Affairs",
        "Risk Management",
        "Regulatory Compliance"
    ],
    "Customer Success & Support": [
        "Customer Success",
        "Technical Support",
        "Client Services"
    ],
    "Procurement & Supply Chain": [
        "Procurement",
        "Logistics",
        "Supply Planning",
        "Vendor Management"
    ]
}

# ------------------------------------------------------------
# 4. Employee titles
# ------------------------------------------------------------

individual_contributor_titles = {
    "Engineering & Technology": [
        "Software Engineer",
        "Senior Software Engineer",
        "Backend Developer",
        "Frontend Developer",
        "DevOps Engineer",
        "Cloud Engineer",
        "QA Engineer",
        "Platform Engineer"
    ],
    "Data & Analytics": [
        "Data Analyst",
        "Senior Data Analyst",
        "Data Engineer",
        "Senior Data Engineer",
        "Analytics Engineer",
        "Business Intelligence Analyst",
        "Data Scientist"
    ],
    "Sales & Business Development": [
        "Sales Executive",
        "Account Executive",
        "Business Development Executive",
        "Senior Sales Executive",
        "Account Manager"
    ],
    "Marketing": [
        "Marketing Specialist",
        "Digital Marketing Executive",
        "SEO Specialist",
        "Content Strategist",
        "Brand Specialist"
    ],
    "Finance & Accounting": [
        "Financial Analyst",
        "Accountant",
        "Senior Accountant",
        "Finance Associate",
        "Tax Analyst"
    ],
    "Human Resources": [
        "HR Executive",
        "Recruiter",
        "Talent Acquisition Specialist",
        "HR Business Partner",
        "People Operations Specialist"
    ],
    "Product Management": [
        "Product Analyst",
        "Product Associate",
        "Product Designer",
        "Product Owner",
        "Business Analyst"
    ],
    "Security & IT": [
        "IT Support Engineer",
        "Security Analyst",
        "Systems Administrator",
        "Network Engineer",
        "Cybersecurity Analyst"
    ],
    "Operations": [
        "Operations Analyst",
        "Operations Executive",
        "Process Analyst",
        "Operations Specialist"
    ],
    "Legal & Compliance": [
        "Compliance Analyst",
        "Legal Associate",
        "Risk Analyst",
        "Legal Counsel"
    ],
    "Customer Success & Support": [
        "Customer Success Associate",
        "Support Specialist",
        "Customer Experience Analyst",
        "Client Success Executive"
    ],
    "Procurement & Supply Chain": [
        "Procurement Analyst",
        "Supply Chain Analyst",
        "Logistics Coordinator",
        "Purchasing Specialist"
    ]
}

manager_titles = {
    "Engineering & Technology": [
        "Engineering Manager",
        "Senior Engineering Manager",
        "Software Development Manager"
    ],
    "Data & Analytics": [
        "Data Engineering Manager",
        "Analytics Manager",
        "Data Science Manager"
    ],
    "Sales & Business Development": [
        "Sales Manager",
        "Regional Sales Manager",
        "Business Development Manager"
    ],
    "Marketing": [
        "Marketing Manager",
        "Digital Marketing Manager",
        "Brand Manager"
    ],
    "Finance & Accounting": [
        "Finance Manager",
        "Accounting Manager",
        "Financial Planning Manager"
    ],
    "Human Resources": [
        "HR Manager",
        "Talent Acquisition Manager",
        "People Operations Manager"
    ],
    "Product Management": [
        "Product Manager",
        "Senior Product Manager",
        "Group Product Manager"
    ],
    "Security & IT": [
        "IT Manager",
        "Security Manager",
        "Infrastructure Manager"
    ],
    "Operations": [
        "Operations Manager",
        "Process Improvement Manager",
        "Business Operations Manager"
    ],
    "Legal & Compliance": [
        "Compliance Manager",
        "Legal Operations Manager",
        "Risk Manager"
    ],
    "Customer Success & Support": [
        "Customer Success Manager",
        "Support Manager",
        "Client Services Manager"
    ],
    "Procurement & Supply Chain": [
        "Procurement Manager",
        "Supply Chain Manager",
        "Logistics Manager"
    ]
}

team_lead_titles = {
    "Engineering & Technology": [
        "Technical Lead",
        "Development Lead",
        "Engineering Team Lead"
    ],
    "Data & Analytics": [
        "Data Team Lead",
        "Analytics Lead",
        "Senior Data Lead"
    ],
    "Sales & Business Development": [
        "Sales Team Lead",
        "Business Development Lead"
    ],
    "Marketing": [
        "Marketing Team Lead",
        "Campaign Lead"
    ],
    "Finance & Accounting": [
        "Finance Team Lead",
        "Accounting Lead"
    ],
    "Human Resources": [
        "HR Team Lead",
        "Recruitment Lead"
    ],
    "Product Management": [
        "Product Lead",
        "Product Team Lead"
    ],
    "Security & IT": [
        "IT Team Lead",
        "Security Team Lead"
    ],
    "Operations": [
        "Operations Team Lead",
        "Process Lead"
    ],
    "Legal & Compliance": [
        "Compliance Team Lead",
        "Legal Team Lead"
    ],
    "Customer Success & Support": [
        "Customer Support Lead",
        "Customer Success Team Lead"
    ],
    "Procurement & Supply Chain": [
        "Procurement Team Lead",
        "Supply Chain Lead"
    ]
}

# ------------------------------------------------------------
# 5. Department head title generator
# ------------------------------------------------------------

def get_department_head_title(department, rng):
    return rng.choice([
        f"Head of {department}",
        f"Director of {department}",
        f"Senior Director of {department}"
    ])

# ------------------------------------------------------------
# 6. Read company metadata
# ------------------------------------------------------------

companies = (
    spark.table(COMPANY_TABLE)
    .select(
        "company_id",
        "employee_count",
        "org_structure"
    )
    .orderBy("company_id")
    .collect()
)

employee_records = []
ground_truth_records = []

# ------------------------------------------------------------
# 7. Employee creation helper
# ------------------------------------------------------------

def add_employee(
    company_id,
    employee_number,
    title,
    department,
    business_unit,
    team_id,
    team_name,
    manager_id
):

    employee_id = (
        f"{company_id}_EMP_{employee_number:04d}"
    )

    employee_records.append((
        company_id,
        employee_id,
        title,
        department,
        business_unit,
        team_id,
        team_name
    ))

    ground_truth_records.append((
        company_id,
        employee_id,
        manager_id
    ))

    return employee_id

# ------------------------------------------------------------
# 8. Generate companies and hierarchies
# ------------------------------------------------------------

for company_index, company in enumerate(companies, start=1):

    company_id = company["company_id"]
    employee_count = company["employee_count"]
    structure = company["org_structure"]

    rng = random.Random(10000 + company_index)

    employee_number = 1

    # CEO
    ceo_id = add_employee(
        company_id,
        employee_number,
        "Chief Executive Officer",
        "Executive",
        None,
        None,
        None,
        None
    )

    employee_number += 1

    executive_ids = {}
    division_ids = {}

    # Functional and Hybrid leadership
    if structure in ["Functional", "Hybrid"]:

        for code, title in executive_titles.items():

            executive_ids[code] = add_employee(
                company_id,
                employee_number,
                title,
                "Executive",
                None,
                None,
                None,
                ceo_id
            )

            employee_number += 1

    # Divisional leadership
    if structure == "Divisional":

        for division_name in divisions:

            division_ids[division_name] = add_employee(
                company_id,
                employee_number,
                f"President - {division_name} Division",
                "Executive",
                division_name,
                None,
                None,
                ceo_id
            )

            employee_number += 1

    # --------------------------------------------------------
    # 9. Allocate employee population across departments
    # --------------------------------------------------------

    leadership_count = employee_number - 1

    remaining_employees = (
        employee_count - leadership_count
    )

    # Minimum 6 employees per department
    department_counts = {
        department: 6
        for department in departments
    }

    extra_employees = (
        remaining_employees - sum(department_counts.values())
    )

    for _ in range(extra_employees):
        selected_department = rng.choice(departments)
        department_counts[selected_department] += 1

    # --------------------------------------------------------
    # 10. Generate department and team hierarchy
    # --------------------------------------------------------

    for department in departments:

        dept_count = department_counts[department]

        # Determine business unit and department head
        if structure == "Divisional":

            business_unit = department_division[department]

            department_parent_id = division_ids[business_unit]

        elif structure in ["Functional", "Hybrid"]:

            business_unit = None

            executive_code = department_executives[department]

            department_parent_id = executive_ids[executive_code]

        else:

            business_unit = None
            department_parent_id = ceo_id

        department_head_id = add_employee(
            company_id,
            employee_number,
            get_department_head_title(department, rng),
            department,
            business_unit,
            None,
            None,
            department_parent_id
        )

        employee_number += 1

        # ----------------------------------------------------
        # 11. Decide how many teams the department has
        # ----------------------------------------------------

        available_team_names = department_teams[department]

        team_count = min(
            len(available_team_names),
            max(1, math.ceil((dept_count - 1) / 8))
        )

        selected_teams = rng.sample(
            available_team_names,
            team_count
        )

        # At least 3 employees per team:
        # Non-flat: Manager + Lead + IC
        # Flat: Lead + 2 ICs
        team_sizes = [3] * team_count

        extra_team_employees = (
            dept_count - 1 - (3 * team_count)
        )

        for _ in range(extra_team_employees):
            team_sizes[rng.randrange(team_count)] += 1

        # ----------------------------------------------------
        # 12. Generate each team
        # ----------------------------------------------------

        for team_index, team_name in enumerate(selected_teams):

            team_id = (
                f"{company_id}_"
                f"D{departments.index(department) + 1:02d}_"
                f"T{team_index + 1:02d}"
            )

            team_size = team_sizes[team_index]

            if structure == "Flat":

                # Flat:
                # Department Head -> Team Lead -> ICs

                lead_id = add_employee(
                    company_id,
                    employee_number,
                    rng.choice(team_lead_titles[department]),
                    department,
                    business_unit,
                    team_id,
                    team_name,
                    department_head_id
                )

                employee_number += 1

                for _ in range(team_size - 1):

                    add_employee(
                        company_id,
                        employee_number,
                        rng.choice(
                            individual_contributor_titles[department]
                        ),
                        department,
                        business_unit,
                        team_id,
                        team_name,
                        lead_id
                    )

                    employee_number += 1

            else:

                # Functional / Divisional / Hybrid:
                # Department Head -> Team Manager -> Team Lead -> ICs

                team_manager_id = add_employee(
                    company_id,
                    employee_number,
                    rng.choice(manager_titles[department]),
                    department,
                    business_unit,
                    team_id,
                    team_name,
                    department_head_id
                )

                employee_number += 1

                team_lead_id = add_employee(
                    company_id,
                    employee_number,
                    rng.choice(team_lead_titles[department]),
                    department,
                    business_unit,
                    team_id,
                    team_name,
                    team_manager_id
                )

                employee_number += 1

                for _ in range(team_size - 2):

                    add_employee(
                        company_id,
                        employee_number,
                        rng.choice(
                            individual_contributor_titles[department]
                        ),
                        department,
                        business_unit,
                        team_id,
                        team_name,
                        team_lead_id
                    )

                    employee_number += 1

    # --------------------------------------------------------
    # 13. Exact employee count enforcement
    # --------------------------------------------------------

    generated_count = employee_number - 1

    if generated_count != employee_count:

        raise ValueError(
            f"{company_id}: expected {employee_count}, "
            f"generated {generated_count}"
        )

# ------------------------------------------------------------
# 14. Create employee DataFrame
# ------------------------------------------------------------

employee_schema = StructType([
    StructField("company_id", StringType(), False),
    StructField("employee_id", StringType(), False),
    StructField("job_title", StringType(), True),
    StructField("department", StringType(), True),
    StructField("business_unit", StringType(), True),
    StructField("team_id", StringType(), True),
    StructField("team_name", StringType(), True)
])

employees_df = spark.createDataFrame(
    employee_records,
    schema=employee_schema
)

# ------------------------------------------------------------
# 15. Create isolated ground truth
# ------------------------------------------------------------

ground_truth_schema = StructType([
    StructField("company_id", StringType(), False),
    StructField("employee_id", StringType(), False),
    StructField("actual_manager_id", StringType(), True)
])

ground_truth_df = spark.createDataFrame(
    ground_truth_records,
    schema=ground_truth_schema
)

# ------------------------------------------------------------
# 16. Save Bronze tables
# ------------------------------------------------------------

(
    employees_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(EMPLOYEE_TABLE)
)

(
    ground_truth_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GROUND_TRUTH_TABLE)
)

# ------------------------------------------------------------
# 17. Summary
# ------------------------------------------------------------

print("TEAM-BASED EMPLOYEE GENERATION COMPLETED")
print("========================================")

print(f"Companies: {len(companies):,}")
print(f"Employees: {len(employee_records):,}")
print(f"Ground truth records: {len(ground_truth_records):,}")
print(f"Actual reporting relationships: {len(companies) and len(ground_truth_records) - len(companies):,}")

print("\nDEPARTMENT DISTRIBUTION")

display(
    employees_df
    .groupBy("department")
    .count()
    .orderBy(F.desc("count"))
)

print("\nTEAM DISTRIBUTION")

display(
    employees_df
    .groupBy("department", "team_name")
    .count()
    .orderBy("department", "team_name")
)

print("\nSAMPLE TEAM-BASED EMPLOYEES")

display(employees_df.limit(20))

print("\nSAMPLE GROUND TRUTH")

display(ground_truth_df.limit(20))

In [0]:
# ============================================================
# FIRMABLE - SYNTHETIC DATA GENERATION
# CELL 3: MESSY SOURCE DATA WITH TEAM INFORMATION
# ============================================================

from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE = f"{CATALOG}.firmable_v2_bronze"

EMPLOYEE_TABLE = f"{BRONZE}.employees_raw"
MESSY_TABLE = f"{BRONZE}.employees_ingestion_raw"

source_df = spark.table(EMPLOYEE_TABLE)

messy_df = (
    source_df
    .withColumn(
        "_title_hash",
        F.pmod(
            F.xxhash64("employee_id", F.lit("title")),
            F.lit(100)
        )
    )
    .withColumn(
        "_dept_hash",
        F.pmod(
            F.xxhash64("employee_id", F.lit("department")),
            F.lit(100)
        )
    )
    .withColumn(
        "_unit_hash",
        F.pmod(
            F.xxhash64("employee_id", F.lit("unit")),
            F.lit(100)
        )
    )
)

# Job-title noise
messy_df = messy_df.withColumn(
    "job_title",
    F.when(
        F.col("_title_hash") < 1,
        F.lit(None).cast("string")
    )
    .when(
        F.col("_title_hash").between(1, 3),
        F.upper("job_title")
    )
    .when(
        F.col("_title_hash").between(4, 7),
        F.concat(F.lit("  "), F.col("job_title"), F.lit("  "))
    )
    .when(
        F.col("_title_hash").between(8, 10),
        F.regexp_replace("job_title", r"\bSenior\b", "Sr.")
    )
    .otherwise(F.col("job_title"))
)

# Department noise
messy_df = messy_df.withColumn(
    "department",
    F.when(
        F.col("_dept_hash") < 2,
        F.lit(None).cast("string")
    )
    .when(
        F.col("_dept_hash").between(2, 4),
        F.upper("department")
    )
    .when(
        F.col("_dept_hash").between(5, 6),
        F.regexp_replace("department", " & ", " and ")
    )
    .when(
        F.col("_dept_hash").between(7, 8),
        F.concat(F.lit(" "), F.col("department"), F.lit(" "))
    )
    .otherwise(F.col("department"))
)

# Business-unit noise
messy_df = messy_df.withColumn(
    "business_unit",
    F.when(
        F.col("business_unit").isNotNull() &
        (F.col("_unit_hash") == 0),
        F.lit(None).cast("string")
    )
    .otherwise(F.col("business_unit"))
)

# Keep team IDs and team names intact.
# They are separate organizational attributes, not manager IDs.

messy_df = messy_df.select(
    "company_id",
    "employee_id",
    "job_title",
    "department",
    "business_unit",
    "team_id",
    "team_name"
)

(
    messy_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MESSY_TABLE)
)

print("TEAM-AWARE MESSY INGESTION CREATED")
print("==================================")

print(f"Source employees: {source_df.count():,}")
print(f"Ingestion records: {messy_df.count():,}")

print("\nMISSING VALUES")

display(
    messy_df.select(
        F.count("*").alias("total_employees"),
        F.sum(
            F.when(F.col("job_title").isNull(), 1).otherwise(0)
        ).alias("missing_titles"),
        F.sum(
            F.when(F.col("department").isNull(), 1).otherwise(0)
        ).alias("missing_departments"),
        F.sum(
            F.when(F.col("business_unit").isNull(), 1).otherwise(0)
        ).alias("missing_business_units"),
        F.sum(
            F.when(F.col("team_id").isNull(), 1).otherwise(0)
        ).alias("missing_team_ids"),
        F.sum(
            F.when(F.col("team_name").isNull(), 1).otherwise(0)
        ).alias("missing_team_names")
    )
)

print("\nSAMPLE MESSY TEAM RECORDS")

display(
    messy_df.filter(
        F.col("job_title").isNull() |
        F.col("department").isNull()
    ).limit(20)
)